# Exploring a brewing yeast genome — `BRQ` variant graph

This notebook walks through the core widget API using a real genomic example:
an 8.5 kb fragment of *S. cerevisiae* chrIV (positions 1,508,000–1,516,500)
containing three genes relevant to brewing:

| Gene | Function | Brewing relevance |
|------|----------|-------------------|
| **STL1** | Glycerol proton symporter | Osmostress response |
| **PAD1** | Phenylacrylic acid decarboxylase | Phenolic off-flavour pathway |
| **FDC1** | Ferulic acid decarboxylase | Converts ferulic acid → 4-vinyl guaiacol |

**BRQ** is a brewing yeast strain.  We load the S288c reference sequence and
apply BRQ variant calls from a VCF to build a graph that encodes
both haplotypes simultaneously.

**Workflow:**
1. Import the reference FASTA and BRQ VCF
2. Plot the BRQ variant graph
3. Load gene annotations from the GFF3 — establish gene boundaries first
4. Add and remove inline annotations on the canvas
5. Search for TATA boxes against the annotated graph
6. Build a k-mer index, search again — same call, faster
7. Highlight multiple motifs in distinct colours
8. Compare the same motif across reference and BRQ block groups

## Setup

The fixture files (`yeast_fragment.fa`, `yeast_variants.vcf`, `yeast_fragment.gff3`)
live in the same directory as this notebook.  We create a fresh repository in a
temporary directory so repeated runs don't accumulate state.

In [1]:
import pathlib
import tempfile

import gen

EXAMPLES_DIR = pathlib.Path(".").resolve()
FASTA = EXAMPLES_DIR / "yeast_fragment.fa"
VCF   = EXAMPLES_DIR / "yeast_variants.vcf"
GFF3  = EXAMPLES_DIR / "yeast_fragment.gff3"

assert FASTA.exists(), f"Missing: {FASTA}"
assert VCF.exists(),   f"Missing: {VCF}"
assert GFF3.exists(),  f"Missing: {GFF3}"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-yeast-"))
print(f"Working in {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
repo.import_fasta(str(FASTA), sample='reference')
repo.update_with_vcf(str(VCF), parent_sample='reference')  # creates BRQ sample

bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) imported")
for b in bgs:
    print(f"  {b.sample_name:15s}  {b.name}")

bg_brq = next(b for b in bgs if b.sample_name == 'BRQ')
bg_ref = next(b for b in bgs if b.sample_name == 'reference')

Working in /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-yeast-q1j_konr
2 block group(s) imported
  reference        pad1_fdc1_region
  BRQ              pad1_fdc1_region


## Plot the BRQ variant graph

The BRQ graph encodes both the S288c reference path and the BRQ-specific
variant path.  Nodes shared between both haplotypes are displayed once;
divergent nodes appear as bubbles.

Calling `fig.show_path()` draws the active path (BRQ) on top of the graph
as a coloured ribbon, making it easy to trace the variant haplotype through
the shared topology.

In [2]:
fig_path = bg_brq.plot()
fig_path.show_path()

## Gene annotations from GFF3

`add_annotation_track_file()` loads a GFF3 or BED file and projects each
feature onto the graph as a coloured bar in a track panel below the canvas.

We filter to `gene` rows only to show STL1, PAD1, and FDC1 without the
redundant CDS and chromosome entries.  Establishing gene boundaries first
makes it easier to interpret search results in later cells.

In [3]:
fig_genes = bg_brq.plot(rows=24)

fig_genes.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "CDS",
    name="genes",
)

print("Track panels:", fig_genes.annotation_tracks())



Track panels: ['genes']


## Multiple annotation layers

Call `add_annotation_track_file` or `add_annotation_track` multiple times
to stack layers.  Each layer gets its own colour row.

Below we stack the GFF3 gene track with a search-derived TATA box track.
With both visible you can see which TATA boxes fall upstream of each gene.

In [4]:
tata_hits = bg_brq.search("tataaa")
print(f"TATA boxes found: {len(tata_hits)}")

fig_stacked = bg_brq.plot(rows=28)

fig_stacked.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
fig_stacked.add_annotation_track(
    [gen.Annotation(h, name=f"TATA:{i}") for i, h in enumerate(tata_hits)],
    name="TATA boxes",
)

print("Tracks:", fig_stacked.annotation_tracks())

for i, h in enumerate(tata_hits[:3]):
    fig_genes.show(h)
fig_genes.go_to(tata_hits[0].start())

TATA boxes found: 10


Tracks: ['genes', 'TATA boxes']


## Removing individual track layers

`remove_annotation_track(name)` removes a single layer by name.
The widget re-renders immediately.

In [5]:
fig_remove_track = bg_brq.plot(rows=28)

fig_remove_track.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
fig_remove_track.add_annotation_track(
    [gen.Annotation(h, name=f"TATA:{i}") for i, h in enumerate(tata_hits)],
    name="TATA boxes",
)

print("Before:", fig_remove_track.annotation_tracks())
fig_remove_track.remove_annotation_track("TATA boxes")
print("After: ", fig_remove_track.annotation_tracks())

Before: ['genes', 'TATA boxes']
After:  ['genes']


## Inline annotations

`add_annotation()` renders an annotation **directly on the graph canvas**.
Each span is tinted with an accent colour and its name is placed just below
the span's bounding box.

Here we combine a gene track panel with inline annotations for specific
sequence motifs, so you can see both the gene boundaries and the exact
positions of the TATA boxes and start codon contexts within them.

In [6]:
start_hits = bg_brq.search("atgaag")

fig_inline = bg_brq.plot(rows=28)
fig_inline.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "CDS",
    name="CDS",
)
for i, h in enumerate(start_hits):
    fig_inline.add_annotation(gen.Annotation(h, name=f"ATG:{i}"))

print("Inline annotations:", fig_inline.inline_annotations())

# Navigate to first start codon context
if start_hits:
    fig_inline.go_to(start_hits[0].start())

Inline annotations: ['ATG:0', 'ATG:1', 'ATG:2', 'ATG:3', 'ATG:4']


In [7]:

start_hits[0]

GraphLocus(1540a1dd[0..454]+5 → 1540a1dd[0..454]+11, 1 blocks)

### Removing inline annotations

`remove_annotation(name)` removes all inline annotations with that name.
`clear_all_inline_annotations()` removes all of them at once.
`clear_all_annotations()` removes both track panels and inline annotations.

In [8]:
fig_inline_rm = bg_brq.plot(rows=28)
fig_inline_rm.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for i, h in enumerate(tata_hits):
    fig_inline_rm.add_annotation(gen.Annotation(h, name=f"TATA:{i}"))
for i, h in enumerate(start_hits):
    fig_inline_rm.add_annotation(gen.Annotation(h, name=f"ATG:{i}"))

print("Before:", fig_inline_rm.inline_annotations())

for i in range(len(start_hits)):
    fig_inline_rm.remove_annotation(f"ATG:{i}")

print("After: ", fig_inline_rm.inline_annotations())

Before: ['TATA:0', 'TATA:1', 'TATA:2', 'TATA:3', 'TATA:4', 'TATA:5', 'TATA:6', 'TATA:7', 'TATA:8', 'TATA:9', 'ATG:0', 'ATG:1', 'ATG:2', 'ATG:3', 'ATG:4']
After:  ['TATA:0', 'TATA:1', 'TATA:2', 'TATA:3', 'TATA:4', 'TATA:5', 'TATA:6', 'TATA:7', 'TATA:8', 'TATA:9']


## Search without an index

`bg.search()` works immediately — no index required.  Without a `.bin` file
it falls back to a full graph scan.

We search for **`TATAAA`**, the canonical TATA-box motif.  This hexamer sits
~25–30 bp upstream of transcription start sites in yeast.  With the gene
track loaded below the graph you can see at a glance which hits fall
upstream of STL1, PAD1, and FDC1.

**Case insensitivity** — search is case-insensitive by default (`case_sensitive=False`).
Pass `case_sensitive=True` to require exact case.

In [9]:
TATA_BOX = "tataaa"

tata_hits = bg_brq.search(TATA_BOX)
print(f"Found {len(tata_hits)} TATA box(es) (full scan)")
for m in tata_hits:
    print(" ", m)

fig_scan = bg_brq.plot(rows=24)
fig_scan.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "gene",
    name="genes",
)
for m in tata_hits:
    fig_scan.show(m)
fig_scan.go_to(tata_hits[0].start())

Found 10 TATA box(es) (full scan)
  GraphLocus(49d4da16[0..1]+0 → 1540a1dd[3014..3041]+5, 2 blocks)
  GraphLocus(1540a1dd[1793..1993]+110 → 1540a1dd[1793..1993]+116, 1 blocks)
  GraphLocus(1540a1dd[1793..1993]+121 → 1540a1dd[1793..1993]+127, 1 blocks)
  GraphLocus(1540a1dd[4433..4553]+118 → 1540a1dd[4554..4589]+3, 3 blocks)
  GraphLocus(1540a1dd[5504..5649]+93 → 1540a1dd[5504..5649]+99, 1 blocks)
  GraphLocus(1540a1dd[5959..6235]+62 → 1540a1dd[5959..6235]+68, 1 blocks)
  GraphLocus(1540a1dd[5959..6235]+107 → 1540a1dd[5959..6235]+113, 1 blocks)
  GraphLocus(1540a1dd[7662..7864]+0 → 1540a1dd[7662..7864]+6, 1 blocks)
  GraphLocus(1540a1dd[7662..7864]+48 → 1540a1dd[7662..7864]+54, 1 blocks)
  GraphLocus(1540a1dd[7865..7942]+65 → 1540a1dd[7865..7942]+71, 1 blocks)


## Build a seed index

`repo.build_index()` writes a junction-aware k-mer index to
`.gen/search_index/<id>.bin`.  Once present, `search()` loads it
automatically — the call is identical, just faster.

- Omit `bgs` to index all block groups at once.
- `k` defaults to 16; `k=8` works well for short motifs like TATA boxes.
- `bg.build_index()` indexes a single block group in place.

**Case-sensitive indexing** — Pass `case_sensitive=True` to build a
case-sensitive index.  Index and query must use the same value; a mismatch
causes a fallback to full-scan (slower but still correct).

In [10]:
repo.build_index(k=8)
print("Index built.")

Index built.


## Search again — now with the index

Identical call.  The index is picked up automatically; search is
seed-extended rather than a full scan.  The results are the same —
the index only changes speed, not correctness.

In [11]:
tata_hits = bg_brq.search(TATA_BOX)
print(f"Found {len(tata_hits)} TATA box(es) (indexed)")

fig_indexed = bg_brq.plot(rows=24)
fig_indexed.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for m in tata_hits:
    fig_indexed.show(m)

Found 10 TATA box(es) (indexed)


## Highlight multiple motifs

`show()` auto-assigns the next unused theme accent colour when called
without an explicit `color` argument — each new motif gets a distinct
colour automatically.

Here we highlight two promoter elements against the gene track:
- **TATA box** (`TATAAA`) — core promoter element ~30 bp upstream of TSS
- **Start codon context** (`ATGAAG`) — methionine-lysine N-terminal motif
  found at the start of STL1 and other genes in this region

In [12]:
MOTIFS = [
    "tataaa",  # TATA box
    "atgaag",  # start codon context (STL1 N-terminus)
]

fig_auto = bg_brq.plot(rows=24)
fig_auto.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "gene",
    name="genes",
)
for query in MOTIFS:
    hits = bg_brq.search(query)
    for m in hits:
        fig_auto.show(m)  # colour auto-assigned from theme
        break
    print(f"{query!r:12s}  {len(hits)} hit(s)")

fig_auto.go_to(tata_hits[0].start())

'tataaa'      10 hit(s)
'atgaag'      5 hit(s)


In [13]:
# Override colours explicitly.
MOTIFS_COLORED = [
    ("tataaa", "#f9e2af"),  # yellow  — TATA box
    ("atgaag", "#94e2d5"),  # teal    — start codon context
]

fig_explicit = bg_brq.plot(rows=24)
fig_explicit.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for query, color in MOTIFS_COLORED:
    hits = bg_brq.search(query)
    for m in hits:
        fig_explicit.show(m, color)
    print(f"{query!r:12s}  {len(hits)} hit(s)  [{color}]")

'tataaa'      10 hit(s)  [#f9e2af]
'atgaag'      5 hit(s)  [#94e2d5]


## Compare reference and BRQ block groups

`repo.search()` without a `bgs` argument searches every block group.
This lets you see whether a motif is present in the reference but absent
in BRQ (or vice versa), revealing strain-specific sequence differences.

TATA boxes are conserved regulatory elements — finding the same count in
both haplotypes is expected.  A discrepancy would suggest a variant
disrupts a promoter in the BRQ strain.

In [14]:
hits_by_bg = repo.search("TATAAA")

print(f"'TATAAA' across {len(bgs)} block group(s):")
for b, hits in hits_by_bg:
    print(f"  {b.sample_name:15s}  {len(hits)} hit(s)")

hits_map = {b.sample_name: hits for b, hits in hits_by_bg}

fig_brq_tata = bg_brq.plot(rows=24)
fig_brq_tata.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "gene",
    name="genes",
)
for m in hits_map.get('BRQ', []):
    fig_brq_tata.show(m, "#f9e2af")
print("BRQ:")
fig_brq_tata.go_to(hits_map.get('BRQ', [])[0].start())

'TATAAA' across 2 block group(s):
  reference        8 hit(s)
  BRQ              10 hit(s)


BRQ:


In [15]:
fig_ref_tata = bg_ref.plot(rows=24)
fig_ref_tata.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for m in hits_map.get('reference', []):
    fig_ref_tata.show(m, "#f9e2af")
print("Reference:")

Reference:


## Regulatory motif analysis — HOG pathway and stress response

The three genes in this region sit under distinct regulatory programs.
We search for well-characterised cis-elements and map them against the gene track.

| Motif class | TF | Canonical sequence | Strand |
|-------------|----|--------------------|--------|
| **Hot1 / HoRE core** | Hot1 (HOG activator) | `GGGACAAA` | both |
| **HoRE broader** | Hot1 | `TGGG[AT]?CAA[AT]?G` (9 variants) | both |
| **CATTTGGC-like repeat** | — | `CATTTGGC` | both |
| **Sko1 / CRE canonical** | Sko1 (HOG repressor/activator) | `TGACGTCA` | palindrome |
| **Sko1-like loose** | Sko1 | `[AT]TGACGTA[CT]` (4 variants) | both |
| **STRE** | Msn2 / Msn4 | `AGGGG` / `CCCCT` | both |

Because the search API takes exact strings, degenerate patterns and reverse complements
are enumerated manually below.  Each search call is case-insensitive by default.

**STL1** is a primary HOG target (Hot1 + Sko1 sites expected in its promoter).
**PAD1** and **FDC1** are less HOG-specific; STRE sites are credible but should be
treated as candidate hits unless supported by ChIP/YEASTRACT evidence.

In [16]:
# --- Hot1 / HoRE core: GGGACAAA (RC: TTTGTCCC) ---
hore_core_hits = (
    bg_brq.search("gggacaaa") +
    bg_brq.search("tttgtccc")
)

# --- HoRE broader: TGGG[AT]?CAA[AT]?G — enumerate 9 fwd variants + 9 RC ---
_hore_fwd = [
    "tgggcaag",   "tgggacaag",   "tgggtcaag",
    "tgggcaaag",  "tgggcaatg",
    "tgggacaaag", "tgggacaatg",
    "tgggtcaaag", "tgggtcaatg",
]
_hore_rev = [
    "cttgccca",   "cttgtccca",   "cttgaccca",
    "ctttgccca",  "cattgccca",
    "ctttgtccca", "cattgtccca",
    "ctttgaccca", "cattgaccca",
]
hore_broad_hits = []
for m in _hore_fwd + _hore_rev:
    hore_broad_hits += bg_brq.search(m)

# --- CATTTGGC-like repeat (RC: GCCAAATG) ---
catttggc_hits = (
    bg_brq.search("catttggc") +
    bg_brq.search("gccaaatg")
)

# --- Sko1 / CRE canonical: TGACGTCA — palindrome, single search ---
cre_hits = bg_brq.search("tgacgtca")

# --- Sko1-like loose: [AT]TGACGTA[CT] and RC [AG]TACGTCA[AT] ---
_sko1_fwd = ["atgacgtat", "atgacgtac", "ttgacgtat", "ttgacgtac"]
_sko1_rev = ["atacgtcat", "gtacgtcat", "atacgtcaa", "gtacgtcaa"]
sko1_loose_hits = []
for m in _sko1_fwd + _sko1_rev:
    sko1_loose_hits += bg_brq.search(m)

# --- STRE: AGGGG (RC: CCCCT) ---
stre_hits = (
    bg_brq.search("agggg") +
    bg_brq.search("cccct")
)

print(f"HoRE core  (GGGACAAA / TTTGTCCC):          {len(hore_core_hits):3d} hit(s)")
print(f"HoRE broad (TGGG[AT]?CAA[AT]?G, both str): {len(hore_broad_hits):3d} hit(s)")
print(f"CATTTGGC-like repeat (both strands):        {len(catttggc_hits):3d} hit(s)")
print(f"CRE / Sko1 canonical TGACGTCA (palindrome): {len(cre_hits):3d} hit(s)")
print(f"Sko1-like loose [AT]TGACGTA[CT] (both str): {len(sko1_loose_hits):3d} hit(s)")
print(f"STRE AGGGG / CCCCT (both strands):          {len(stre_hits):3d} hit(s)")

HoRE core  (GGGACAAA / TTTGTCCC):            0 hit(s)
HoRE broad (TGGG[AT]?CAA[AT]?G, both str):   0 hit(s)
CATTTGGC-like repeat (both strands):          0 hit(s)
CRE / Sko1 canonical TGACGTCA (palindrome):   0 hit(s)
Sko1-like loose [AT]TGACGTA[CT] (both str):   0 hit(s)
STRE AGGGG / CCCCT (both strands):            1 hit(s)


### BRQ — all regulatory motifs overlaid

Each motif class gets a distinct colour.  The gene track below lets you see
immediately which hits fall in promoter-proximal positions (upstream of STL1,
PAD1, FDC1) versus within coding regions.

| Colour | Motif |
|--------|-------|
| Yellow `#f9e2af` | HoRE core `GGGACAAA` |
| Peach `#fab387` | HoRE broad `TGGG[AT]?CAA[AT]?G` |
| Mauve `#cba6f7` | CRE / Sko1 canonical `TGACGTCA` |
| Sky `#89dceb` | Sko1-like loose |
| Green `#a6e3a1` | STRE `AGGGG/CCCCT` |

Overlapping hits from different classes will paint the later colour on top.

In [17]:
REG_MOTIFS = [
    ("HoRE core",    "#f9e2af", hore_core_hits),
    ("HoRE broad",   "#fab387", hore_broad_hits),
    ("CRE/Sko1",     "#cba6f7", cre_hits),
    ("Sko1 loose",   "#89dceb", sko1_loose_hits),
    ("STRE",         "#a6e3a1", stre_hits),
]

fig_reg_brq = bg_brq.plot(rows=32)
fig_reg_brq.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("	")[2] == "gene",
    name="genes",
)
for label, color, hits in REG_MOTIFS:
    for h in hits:
        fig_reg_brq.show(h, color)
    print(f"{label:15s}  {len(hits):3d} hit(s)  [{color}]")

# Navigate to the STRE hit (the only regulatory motif found)
if stre_hits:
    fig_reg_brq.go_to(stre_hits[0].start())

HoRE core          0 hit(s)  [#f9e2af]
HoRE broad         0 hit(s)  [#fab387]
CRE/Sko1           0 hit(s)  [#cba6f7]
Sko1 loose         0 hit(s)  [#89dceb]
STRE               1 hit(s)  [#a6e3a1]


## Freeze for distribution

`fig.freeze()` captures the current widget state as a static PNG and bakes it
into the `.ipynb` file.  Once frozen:

- All interaction methods (`zoom_in`, `move_by`, `show()`, …) become no-ops.
- The live border indicator on the canvas disappears.
- The PNG renders in GitHub, nbviewer, and other static viewers even without
  the `gen` module installed.

Run all cells above first, then navigate each live widget to the view you want
to preserve.  When you are happy with every figure, run the cell below to
freeze them all at once.

In [18]:
for fig in [fig_path, fig_genes, fig_stacked, fig_remove_track, fig_inline, fig_inline_rm,
            fig_scan, fig_indexed, fig_auto, fig_explicit,
            fig_brq_tata, fig_ref_tata,
            fig_reg_brq]:
    break
    fig.freeze()

## Clear the index

`repo.clear_index()` removes all `.bin` files under `.gen/search_index/`.
Pass `bgs=[...]` to clear only specific entries.  `bg.clear_index()` removes
just that block group's index.

In [19]:
repo.clear_index()
# or: bg_brq.clear_index()
print("Index cleared.")

Index cleared.


## Tips

**Index lifetime** — the index persists on disk between sessions.  Build once
after import; rebuild only if the graph changes.

**Cross-junction matches** — when a query spans a node boundary (e.g. a TATA box
that straddles a variant site), `show()` paints all nodes in the locus.
Check `len(m.nodes) > 1` to detect these.

**Comparing haplotypes** — open separate figures for the reference and BRQ
block groups and apply the same highlights to both.  Differences in hit counts
or positions indicate variant-disrupted features.

**Distributing notebooks** — call `fig.freeze()` before saving to bake a static
PNG into the cell output.  Recipients without the `gen` module see the graph
image instead of a blank widget.